In [1]:
# Chunking splits a document into smaller pieces before indexing it, so
# retrieval can return only the passages relevant to a question instead of
# the whole file.
#
# Chunk by a set number of characters. `chunk_overlap` repeats the tail of one
# chunk at the start of the next, so text cut at a boundary still appears
# complete in at least one chunk.
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))

        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks

In [2]:
# Chunk by sentence: split on sentence-ending punctuation, then group a fixed
# number of sentences per chunk. No sentence is ever cut in half, and the
# overlapping sentence carries context across chunk boundaries.
import re


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))

        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        start_idx += max_sentences_per_chunk - overlap_sentences

    return chunks

In [3]:
# Chunk by section: split on markdown `## ` headings. The boundaries follow
# the document's own structure, so each chunk is a complete, self-contained
# topic — usually the best strategy when the source has meaningful headings.
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [4]:
# The corpus used across all the RAG notebooks: a synthetic interdisciplinary
# report with 14 markdown sections that cross-reference each other.
with open("./report.md", "r") as f:
    text = f.read()

In [5]:
# Character chunks are uniform in size but blind to meaning — note how the
# second chunk starts with the last 20 characters of the first (the overlap).
char_chunks = chunk_by_char(text)
print(f"{len(char_chunks)} chunks of up to 150 characters\n")
print(repr(char_chunks[0]), "\n")
print(repr(char_chunks[1]))

141 chunks of up to 150 characters

'# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n\n## Executive Summary\n\nThis report synthesizes the key findings and ongoing rese' 

"ngs and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the "


In [6]:
# Sentence chunks never split mid-sentence, and consecutive chunks share one
# sentence of overlap.
sentence_chunks = chunk_by_sentence(text)
print(f"{len(sentence_chunks)} chunks of up to 5 sentences\n")
print(sentence_chunks[0])

33 chunks of up to 5 sentences

# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`).


In [7]:
# Section chunks map one-to-one to the report's topics — this is the strategy
# the rest of the RAG notebooks reuse.
section_chunks = chunk_by_section(text)
print(f"{len(section_chunks)} section chunks:\n")

for chunk in section_chunks:
    print(f"- {chunk.splitlines()[0]}")

15 section chunks:

- # **Annual Interdisciplinary Research Review: Cross-Domain Insights**
- Executive Summary
- Table of Contents
- Methodology
- Section 1: Medical Research - Understanding XDR-471 Syndrome
- Section 2: Software Engineering - Project Phoenix Stability Enhancements
- Section 3: Financial Analysis - Q3 Performance and Outlook
- Section 4: Scientific Experimentation - Characterization of Material Composite XT-5
- Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts
- Section 6: Product Engineering - Finalizing Model Zircon-5 Specifications
- Section 7: Historical Research - Re-evaluating the Galveston Accords (1921)
- Section 8: Project Management - Progress on Project Cerberus Phase 2B
- Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update
- Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011
- Future Directions
